#### Validando se o arquivo novo perde ou não dados e se repete as linhas:

In [ ]:
# === Importando Bibliotecas e consumindo a Base de Dados original ===
from openpyxl.styles import PatternFill, Font
from openpyxl import load_workbook
import pandas as pd

# === NOVO (necessário p/ alinhar colunas fora de ordem e criar chaves robustas) ===
import re
import unicodedata


# === CONFIGURAÇÃO ===
arquivo_excel = r".\xlsx\Ativos e Desligados - 09-30-2025.xlsx"  # Primeiro arquivo...
arquivo_comparacao = r".\xlsx\Ativos e Desligados - 06-31-2025.xlsx"  # Segundo arquivo...

# === NOVO: arquivo(s) adicional(is) para inserir linhas novas no final ===
arquivos_adicionais = [
    r".\xlsx\Ativos e Desligados - 12-31-2025.xlsx"
]

saida = r"..\..\..\..\Base de Dados Original\xlsx\Ativos e Desligados\Ativos e Desligados.xlsx"  # Arquivo final...


# === ABRIR O ARQUIVO ORIGINAL (para preservar layout) ===
wb = load_workbook(arquivo_excel)

# === LEITURA DAS PLANILHAS (pandas) ===
pagina1 = 0
pagina2 = 1
pagina3 = 2
pagina4 = 3

df1 = pd.read_excel(arquivo_excel, sheet_name=pagina1)
df2 = pd.read_excel(arquivo_excel, sheet_name=pagina2)
df3 = pd.read_excel(arquivo_excel, sheet_name=pagina3)
df4 = pd.read_excel(arquivo_excel, sheet_name=pagina4)

In [ ]:
# === FUNÇÃO DE LIMPEZA ===
def limpar(df):
    df.columns = df.columns.str.strip().str.lower()
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip().str.lower()
    return df

# === APLICAR LIMPEZA ===
df1, df2, df3, df4 = map(limpar, [df1, df2, df3, df4])

# === FUNÇÃO PARA PREPARAR DATAS ===
def preparar_datas(df, nome_coluna):
    if nome_coluna.lower() in df.columns:
        df[nome_coluna.lower()] = df[nome_coluna.lower()].astype(str)
        df[nome_coluna.lower()] = df[nome_coluna.lower()].str.replace('.', '/', regex=False)
        df[nome_coluna.lower()] = pd.to_datetime(df[nome_coluna.lower()], errors='coerce', dayfirst=True)
        df['ano'] = df[nome_coluna.lower()].dt.year
        df['mes'] = df[nome_coluna.lower()].dt.month
    return df

# === PREPARAR DATAS NAS PÁGINAS ===
df1 = preparar_datas(df1, 'Data_Admissão')
df3 = preparar_datas(df3, 'Data_Admissão')
df2 = preparar_datas(df2, 'Data_Dem')
df4 = preparar_datas(df4, 'Data_Dem')

# === FUNÇÃO DE COMPARAÇÃO (mantida) ===
def comparar_paginas(df_origem, df_destino, nome_origem, nome_destino, coluna_data):
    periodos_comuns = set(zip(df_origem['ano'], df_origem['mes'])) & set(zip(df_destino['ano'], df_destino['mes']))
    df_origem_periodo = df_origem[df_origem[['ano', 'mes']].apply(tuple, axis=1).isin(periodos_comuns)]

    faltando = df_origem_periodo.merge(df_destino, how='left', indicator=True)
    faltando = faltando[faltando['_merge'] == 'left_only'].drop(columns=['_merge'])
    faltando['pagina'] = nome_origem

    print(f"\n🔍 {nome_origem} → {nome_destino}")
    if faltando.empty:
        print("✅ Nenhuma linha diferente encontrada no mesmo período.")
    else:
        print(f"⚠️ {len(faltando)} linha(s) novas em {nome_origem} dentro do mesmo período de {nome_destino}.")

    return faltando


# === ABRIR O ARQUIVO ORIGINAL (mantém formatação e maiúsculas) ===
wb = load_workbook(arquivo_excel)
print("Abas encontradas:", wb.sheetnames)

# === NOVO: helper para limpar dados mantendo cabeçalho ===
def limpar_dados_aba(ws):
    if ws.max_row > 1:
        ws.delete_rows(2, ws.max_row - 1)

# === FUNÇÃO PARA LER UMA ABA COMO TEXTO PURO (sem pandas lower) ===
# === ALTERAÇÃO: agora recebe workbook como parâmetro (pra ler também arquivos adicionais) ===
def ler_aba_como_dataframe(workbook, nome_aba):
    ws = workbook[nome_aba]
    dados = []

    for row in ws.iter_rows(values_only=True):
        dados.append(list(row))

    if not dados:
        return pd.DataFrame()

    colunas = [str(c) if c is not None else f"Col_{i}" for i, c in enumerate(dados[0])]
    df = pd.DataFrame(dados[1:], columns=colunas)
    return df

# === LER AS 4 ABAS COM openpyxl (preserva conteúdo original) ===
df1 = ler_aba_como_dataframe(wb, wb.sheetnames[0])
df2 = ler_aba_como_dataframe(wb, wb.sheetnames[1])
df3 = ler_aba_como_dataframe(wb, wb.sheetnames[2])
df4 = ler_aba_como_dataframe(wb, wb.sheetnames[3])


# === FUNÇÃO PARA UNIR E ATUALIZAR UMA ABA PRESERVANDO LAYOUT E MAIÚSCULAS ===
def unir_e_atualizar(indice_destino, df_a, df_b, nome_destino):
    nome_aba = wb.sheetnames[indice_destino]
    ws = wb[nome_aba]

    colunas_destino = list(df_a.columns)
    for col in df_b.columns:
        if col not in colunas_destino:
            colunas_destino.append(col)

    df_a = df_a.reindex(columns=colunas_destino)
    df_b = df_b.reindex(columns=colunas_destino)

    df_a["origem"] = "1_ou_2"
    df_b["origem"] = "3_ou_4"

    df_unido = pd.concat([df_a, df_b], ignore_index=True)

    # ===============================
    # 🔍 REGRA: MESMO NOME + Nº PESS.
    # ===============================
    if all(col in df_unido.columns for col in ["Nome completo", "Nº pess."]):
        chaves = ["Nome completo", "Nº pess."]

        df_unido["prioridade"] = df_unido["origem"].apply(lambda x: 1 if x == "3_ou_4" else 0)
        df_unido = df_unido.sort_values(by="prioridade", ascending=True)

        antes = len(df_unido)
        df_unido = df_unido.drop_duplicates(subset=chaves, keep="last")
        depois = len(df_unido)

        print(f"🧹 {antes - depois} duplicada(s) removida(s) em {nome_destino} — mantendo versão das páginas 3/4.")

        df_unido = df_unido.drop(columns=["origem", "prioridade"])

    else:
        df_unido = df_unido.drop_duplicates()
        print(f"⚠️ Colunas esperadas ('Nome completo', 'Nº pess.') não encontradas em {nome_destino}.")

    print(f"📄 {nome_destino}: {len(df_unido)} linhas únicas (mantendo dados das páginas 3/4).")

    # Limpa linhas antigas (mantendo cabeçalhos e estilos)
    limpar_dados_aba(ws)

    # Escreve célula por célula
    for r_idx, row in enumerate(df_unido.itertuples(index=False), start=2):
        for c_idx, valor in enumerate(row, start=1):
            ws.cell(row=r_idx, column=c_idx, value=None if pd.isna(valor) else valor)


# === NOVA FUNÇÃO: REMOVER PESSOAS DA ABA 1 QUE TAMBÉM ESTÃO NA ABA 2 ===
def remover_duplicados_entre_abas(indice_aba1, indice_aba2):
    nome_aba1 = wb.sheetnames[indice_aba1]
    nome_aba2 = wb.sheetnames[indice_aba2]

    print(f"\n🔁 Comparando '{nome_aba1}' x '{nome_aba2}' para remover pessoas duplicadas da primeira aba...")

    ws1 = wb[nome_aba1]

    df1_atual = ler_aba_como_dataframe(wb, nome_aba1)
    df2_atual = ler_aba_como_dataframe(wb, nome_aba2)

    chaves = ["Nome completo", "Nº pess."]
    if not all(col in df1_atual.columns for col in chaves) or not all(col in df2_atual.columns for col in chaves):
        print("⚠️ Não foi possível remover duplicados entre abas: colunas 'Nome completo' e 'Nº pess.' não estão presentes nas duas abas.")
        return

    def criar_chave(df):
        return (
            df["Nome completo"].astype(str).str.strip().str.lower()
            + "||"
            + df["Nº pess."].astype(str).str.strip().str.lower()
        )

    df1_atual["_chave"] = criar_chave(df1_atual)
    df2_atual["_chave"] = criar_chave(df2_atual)

    chaves_segunda = set(df2_atual["_chave"].dropna())

    antes = len(df1_atual)
    df1_filtrado = df1_atual[~df1_atual["_chave"].isin(chaves_segunda)].drop(columns=["_chave"])
    removidas = antes - len(df1_filtrado)

    print(f"🧹 {removidas} linha(s) removida(s) da aba '{nome_aba1}' porque a mesma pessoa está na aba '{nome_aba2}' (mantendo só na segunda).")

    limpar_dados_aba(ws1)
    for r_idx, row in enumerate(df1_filtrado.itertuples(index=False), start=2):
        for c_idx, valor in enumerate(row, start=1):
            ws1.cell(row=r_idx, column=c_idx, value=None if pd.isna(valor) else valor)


# ==========================================================
# === NOVO: INSERIR LINHAS NOVAS DE ARQUIVOS ADICIONAIS ====
# ==========================================================

def _norm_txt(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.strip().lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"[^a-z0-9]+", "", s)
    return s

def _concept(col_name: str) -> str:
    n = _norm_txt(col_name)

    # conceitos equivalentes (pra casar Data_Adm vs Data_Admissão etc.)
    if n in {"dataadm", "dataadmissao", "data_admissao", "dataadmiss"}:
        return "dataadmissao"
    if n in {"datadem", "datademissao", "data_dem", "datademiss"}:
        return "datademissao"

    if n in {"npess", "nopess", "numpess", "numeropess", "npessoal"}:
        return "npess"
    if n in {"nomecompleto", "nome"}:
        return "nomecompleto"
    if n == "cpf":
        return "cpf"

    return n

def _build_concept_map(df: pd.DataFrame) -> dict:
    m = {}
    for c in df.columns:
        cc = _concept(c)
        if cc not in m:
            m[cc] = c
    return m

def _clean_digits(v) -> str:
    s = "" if v is None else str(v)
    s = s.replace(".0", "")
    return re.sub(r"\D+", "", s)

def _clean_name(v) -> str:
    s = "" if v is None else str(v)
    s = " ".join(s.strip().split())
    return s.lower()

def _make_key(df: pd.DataFrame) -> pd.Series:
    cmap = _build_concept_map(df)
    c_npess = cmap.get("npess")
    c_cpf = cmap.get("cpf")
    c_nome = cmap.get("nomecompleto")

    if c_npess and c_cpf:
        return df[c_npess].apply(_clean_digits) + "|" + df[c_cpf].apply(_clean_digits)
    if c_npess and c_nome:
        return df[c_npess].apply(_clean_digits) + "|" + df[c_nome].apply(_clean_name)
    if c_cpf and c_nome:
        return df[c_cpf].apply(_clean_digits) + "|" + df[c_nome].apply(_clean_name)

    # fallback (ruim, mas evita quebrar)
    return df.astype(str).agg("|".join, axis=1)

def _escolher_destino(df_extra: pd.DataFrame, wb_principal) -> int:
    # só considera as duas abas finais (0 e 1)
    idxs = [0, 1]
    extra_concepts = set(_concept(c) for c in df_extra.columns)

    melhor_idx = 0
    melhor_score = -1

    for idx in idxs:
        nome_aba = wb_principal.sheetnames[idx]
        df_dest = ler_aba_como_dataframe(wb_principal, nome_aba)
        dest_concepts = set(_concept(c) for c in df_dest.columns)

        inter = len(extra_concepts & dest_concepts)
        score = inter / max(1, len(extra_concepts))

        if score > melhor_score:
            melhor_score = score
            melhor_idx = idx

    return melhor_idx

def inserir_linhas_novas_no_wb(df_extra: pd.DataFrame, idx_destino: int):
    nome_dest = wb.sheetnames[idx_destino]
    ws_dest = wb[nome_dest]

    df_dest = ler_aba_como_dataframe(wb, nome_dest)
    if df_dest.empty:
        df_dest = pd.DataFrame(columns=list(df_extra.columns))

    # 1) Renomeia colunas do extra para casar com o "conceito" das colunas do destino (ex.: Data_Adm -> Data_Admissão)
    cmap_dest = _build_concept_map(df_dest)
    renames = {}
    for col in df_extra.columns:
        c = _concept(col)
        if c in cmap_dest and col != cmap_dest[c]:
            renames[col] = cmap_dest[c]
    if renames:
        df_extra = df_extra.rename(columns=renames)

    # 2) Garante colunas (união, mantendo ordem do destino e adicionando novas no fim)
    colunas = list(df_dest.columns)
    for c in df_extra.columns:
        if c not in colunas:
            colunas.append(c)

    df_dest = df_dest.reindex(columns=colunas)
    df_extra = df_extra.reindex(columns=colunas)

    # 3) Cria chave e insere somente o que não existe ainda
    df_dest["_key"] = _make_key(df_dest)
    df_extra["_key"] = _make_key(df_extra)

    keys_dest = set(df_dest["_key"].dropna())
    novos = df_extra[~df_extra["_key"].isin(keys_dest)].copy()

    print(f"\n➕ Inserindo arquivo adicional em '{nome_dest}': {len(novos)} linha(s) nova(s).")

    df_final = pd.concat([df_dest, novos], ignore_index=True)

    # remove _key antes de escrever
    df_final = df_final.drop(columns=["_key"])

    # 4) Reescreve a aba (mantém cabeçalho)
    limpar_dados_aba(ws_dest)
    for r_idx, row in enumerate(df_final.itertuples(index=False), start=2):
        for c_idx, valor in enumerate(row, start=1):
            ws_dest.cell(row=r_idx, column=c_idx, value=None if pd.isna(valor) else valor)

def integrar_arquivo_adicional(caminho_arquivo: str):
    wb_extra = load_workbook(caminho_arquivo)
    print(f"\n📥 Lendo arquivo adicional: {caminho_arquivo}")
    print("   Abas no adicional:", wb_extra.sheetnames)

    for aba in wb_extra.sheetnames:
        df_extra = ler_aba_como_dataframe(wb_extra, aba)
        if df_extra.empty:
            continue

        idx_destino = _escolher_destino(df_extra, wb)
        inserir_linhas_novas_no_wb(df_extra, idx_destino)


# === UNIR Página 1 + Página 3 ===
unir_e_atualizar(0, df1, df3, "Página 1 + Página 3")

# === UNIR Página 2 + Página 4 ===
unir_e_atualizar(1, df2, df4, "Página 2 + Página 4")

# === NOVO: integrar bases adicionais (ex.: 10-01-2025) antes de remover duplicados entre abas ===
for arq in arquivos_adicionais:
    integrar_arquivo_adicional(arq)

# === AGORA COMPARA ABA 1 x ABA 2 E REMOVE DA 1 QUEM JÁ ESTÁ NA 2 ===
remover_duplicados_entre_abas(0, 1)

In [ ]:
# === RENOMEAR A SEGUNDA ABA PARA 'Desligados' ===
wb.sheetnames  # garante que a lista está atualizada
segunda_aba = wb.worksheets[1]
segunda_aba.title = "Desligados"
print("📝 Aba renomeada: Página 2 + Página 4 → Desligados")

# === REMOVER ABAS 3 e 4 (deixa só as duas primeiras) ===
while len(wb.sheetnames) > 2:
    nome = wb.sheetnames[-1]
    del wb[nome]
    print(f"🗑️ Aba removida: {nome}")

# === SALVAR O NOVO ARQUIVO ===
wb.save(saida)
print(f"\n✅ Arquivo final salvo preservando maiúsculas, conteúdo e visual em: {saida}")